# Práctica guiada · Ajuste de hiperparámetros con tres clasificadores
### Regresión logística · SVM · Naive Bayes

**FP13 · Inteligencia Artificial** — Universidad Autónoma de Guadalajara
**Alumno:** Camilo Tonatiuh Guzmán Arias (Ing. Mecatrónica)

---

### 🏥 El problema clínico
Un servicio de cardiología quiere una herramienta de **apoyo al triaje**: con 11 datos clínicos de rutina se debe decidir si un paciente **probablemente tiene enfermedad cardiaca** y conviene enviarlo a estudios especializados.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVC

In [ ]:
from kagglehub import KaggleDatasetAdapter
df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, 'fedesoriano/heart-failure-prediction', 'heart.csv')
df.head()

In [ ]:
# Corrigiendo los valores numéricos en 0 que no hacen sentido (presión y colesterol)
df['Cholesterol'] = df['Cholesterol'].replace(0, np.nan)
df['RestingBP'] = df['RestingBP'].replace(0, np.nan)

# Separando los datos en X y Y
y = df['HeartDisease']
X = df.drop(columns=['HeartDisease'])

# Aplicando One-Hot Encoding a las columnas de texto para que el modelo las entienda
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Dividiendo los datos en entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

---
## Modelo 1: Regresión Logística

In [ ]:
# Definiendo columnas numéricas a escalar
columnas_escalar = ['Age', 'RestingBP', 'Cholesterol', 'MaxHR', 'Oldpeak']

# Definiendo un column transformer para transformar las variables a escalar 
prep = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputar', SimpleImputer(strategy='median')), # Rellenando nulos con la mediana
            ('escalar', StandardScaler()),
        ]), columnas_escalar),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

# Aplicando pipeline: Preparacion -> Escalamiento -> Modelo Logistico
reglog_base = Pipeline([
    ('prep', prep),
    ('modelo', LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0, max_iter=5000, random_state=42))
])

# Ajustando modelo con valores por defecto
reglog_base.fit(X_train, y_train)

In [ ]:
clases = ['Sano', 'Enfermo']

# Evaluando el modelo de regresión logística
y_pred = reglog_base.predict(X_test)
print(classification_report(y_test, y_pred, target_names=clases))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=clases, cmap='Blues')
plt.title('Matriz de Confusión - Regresión Logística Base')
plt.show()

In [ ]:
metricas = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
rejilla_reglog = {
    'modelo__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'modelo__l1_ratio': [0, 0.5, 1],
    'modelo__class_weight': [None, 'balanced'],
}

mejores_modelos = {}
for metrica in metricas:
    busqueda = GridSearchCV(reglog_base, rejilla_reglog, scoring=metrica, cv=5, n_jobs=-1)
    busqueda.fit(X_train, y_train)
    mejores_modelos[metrica] = busqueda.best_estimator_
    print(f"{metrica} >> {round(busqueda.best_score_, 3)} >> {busqueda.best_params_}")

**Justificación de la métrica seleccionada:**

Nos vamos por la **precisión** como métrica principal porque, pensándolo desde el punto de vista del sistema en la vida real, es clave reducir los falsos positivos. Si por un error del algoritmo mandamos a mucha gente sana a hacerse estudios especializados (que salen bastante caros), solo vamos a saturar la clínica y desperdiciar los recursos del hospital a lo tonto. Es mucho mejor ir a lo seguro y ser precisos con los que de verdad marcamos como enfermos.

---
## Modelo 2: Máquinas de Vectores de Soporte (SVM)

**Notas sobre la corrección del código:**

En la celda original traíamos un clásico error de *copy-paste* donde se mandaba a llamar el entrenamiento de la regresión logística (`reglog_base.fit`) otra vez en lugar de nuestro nuevo modelo SVM. Ya quedó solucionado utilizando `svm_base.fit()`. De paso, le agregué el parámetro `random_state=42` a la inicialización del clasificador `SVC` para obligar a que los resultados sean totalmente reproducibles cada vez que corramos la notebook, lo que nos hace la vida más fácil al momento de depurar.

In [ ]:
# Pipeline para Máquinas de Vectores de Soporte (SVM)
svm_base = Pipeline([
    ('prep', prep),
    ('modelo', SVC(random_state=42)) # Se fijó la semilla para garantizar reproducibilidad
])

# Entrenamiento corregido usando el pipeline del SVM
svm_base.fit(X_train, y_train)

In [ ]:
# Sacando las predicciones del modelo SVM para compararlo
y_pred_svm = svm_base.predict(X_test)

# Imprimiendo el reporte de clasificación para evaluar el desempeño
print('--- Reporte de Clasificación: SVM Base ---')
print(classification_report(y_test, y_pred_svm, target_names=clases))

# Construyendo matriz de confusión para el SVM 
# Le pusimos un mapa de color verde para que a simple vista no se confunda con la logistica
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_svm, display_labels=clases, cmap='Greens')
plt.title('Matriz de Confusión - SVM Base')
plt.show()